In [2]:
import pandas as pd 
import numpy as np

import joblib

pd.set_option("display.max_columns", None)

In [3]:

df = pd.read_csv("../data/retail_store_inventory.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).copy()

grouped = df.groupby(["Store ID", "Product ID"])["Units Sold"]

df["Average Daily Demand"] = grouped.transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df["Demand Std"] = grouped.transform(
    lambda x: x.shift(1).rolling(7).std()
)

print("RetailMind inventory data loaded!")
print("Dataset shape:", df.shape)

display(df.head())


RetailMind inventory data loaded!
Dataset shape: (73100, 17)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Average Daily Demand,Demand Std
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,NaN,NaN
100,2022-01-02,S001,P0001,Groceries,West,116,81,104,92.94,27.95,10,Cloudy,0,30.89,Spring,NaN,NaN
200,2022-01-03,S001,P0001,Electronics,West,154,5,189,5.36,62.70,20,Rainy,0,58.22,Winter,NaN,NaN
300,2022-01-04,S001,P0001,Groceries,South,85,58,193,52.87,77.88,15,Cloudy,1,75.99,Winter,NaN,NaN
400,2022-01-05,S001,P0001,Groceries,South,238,147,37,150.27,28.46,20,Sunny,1,29.40,Winter,NaN,NaN


In [4]:

inventory_df = (
    df.sort_values("Date")
    .groupby(["Store ID", "Product ID"])
    .tail(1)
    .copy()
)

inventory_df = inventory_df[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Category",
        "Inventory Level",
        "Units Sold",
        "Average Daily Demand",
        "Demand Std"
    ]
].copy()

print("Inventory snapshot created!")
print("Total records:", len(inventory_df))

display(inventory_df.head(10))


Inventory snapshot created!
Total records: 100


,Date,Store ID,Product ID,Category,Inventory Level,Units Sold,Average Daily Demand,Demand Std
73049,2024-01-01,S003,P0010,Furniture,413,398,100.571429,65.706055
73097,2024-01-01,S005,P0018,Clothing,278,36,95.428571,129.536169
73025,2024-01-01,S002,P0006,Toys,160,159,124.000000,71.178180
73026,2024-01-01,S002,P0007,Toys,53,2,156.428571,144.250774
73027,2024-01-01,S002,P0008,Furniture,218,43,148.857143,131.538877
73028,2024-01-01,S002,P0009,Electronics,62,31,165.428571,140.876846
73029,2024-01-01,S002,P0010,Toys,419,347,141.000000,149.274914
73030,2024-01-01,S002,P0011,Groceries,156,39,80.714286,77.907454
73031,2024-01-01,S002,P0012,Electronics,193,130,126.571429,80.694604
73032,2024-01-01,S002,P0013,Electronics,472,19,163.142857,93.288850


In [5]:

inventory_df["Stock Coverage"] = (
    inventory_df["Inventory Level"] /
    inventory_df["Average Daily Demand"].replace(0, np.nan)
)

inventory_df["Stock Coverage"] = inventory_df["Stock Coverage"].fillna(0)

print("STOCK COVERAGE CALCULATED!")

display(
    inventory_df[
        [
            "Store ID",
            "Product ID",
            "Inventory Level",
            "Average Daily Demand",
            "Stock Coverage"
        ]
    ].head(10)
)


STOCK COVERAGE CALCULATED!


,Store ID,Product ID,Inventory Level,Average Daily Demand,Stock Coverage
73049,S003,P0010,413,100.571429,4.106534
73097,S005,P0018,278,95.428571,2.913174
73025,S002,P0006,160,124.000000,1.290323
73026,S002,P0007,53,156.428571,0.338813
73027,S002,P0008,218,148.857143,1.464491
73028,S002,P0009,62,165.428571,0.374784
73029,S002,P0010,419,141.000000,2.971631
73030,S002,P0011,156,80.714286,1.932743
73031,S002,P0012,193,126.571429,1.524831
73032,S002,P0013,472,163.142857,2.893170


In [6]:

def classify_inventory(days):
    if days < 2:
        return "Reorder Required"
    elif days <= 7:
        return "Healthy"
    else:
        return "Overstock"

inventory_df["Inventory Status"] = (
    inventory_df["Stock Coverage"].apply(classify_inventory)
)

print("INVENTORY CLASSIFICATION COMPLETED!")

print("\nSTATUS SUMMARY:")
display(inventory_df["Inventory Status"].value_counts())

print("\nINVENTORY DETAILS:")
display(
    inventory_df[
        [
            "Store ID",
            "Product ID",
            "Inventory Level",
            "Average Daily Demand",
            "Stock Coverage",
            "Inventory Status"
        ]
    ].head(15)
)


INVENTORY CLASSIFICATION COMPLETED!

STATUS SUMMARY:


Inventory Status
Reorder Required    53
Healthy             47
Name: count, dtype: int64


INVENTORY DETAILS:


,Store ID,Product ID,Inventory Level,Average Daily Demand,Stock Coverage,Inventory Status
73049,S003,P0010,413,100.571429,4.106534,Healthy
73097,S005,P0018,278,95.428571,2.913174,Healthy
73025,S002,P0006,160,124.000000,1.290323,Reorder Required
73026,S002,P0007,53,156.428571,0.338813,Reorder Required
73027,S002,P0008,218,148.857143,1.464491,Reorder Required
73028,S002,P0009,62,165.428571,0.374784,Reorder Required
73029,S002,P0010,419,141.000000,2.971631,Healthy
73030,S002,P0011,156,80.714286,1.932743,Reorder Required
73031,S002,P0012,193,126.571429,1.524831,Reorder Required
73032,S002,P0013,472,163.142857,2.893170,Healthy


In [7]:

lead_time = 2
target_days = 7
service_factor = 1.65

inventory_df["Safety Stock"] = (
    service_factor *
    inventory_df["Demand Std"] *
    np.sqrt(lead_time)
)

inventory_df["Reorder Point"] = (
    inventory_df["Average Daily Demand"] * lead_time
    + inventory_df["Safety Stock"]
)

inventory_df["Target Stock"] = (
    inventory_df["Average Daily Demand"] * target_days
    + inventory_df["Safety Stock"]
)

inventory_df["Reorder Quantity"] = (
    inventory_df["Target Stock"] -
    inventory_df["Inventory Level"]
).clip(lower=0).round().astype(int)

inventory_df["Reorder Quantity"] = np.where(
    inventory_df["Inventory Status"] == "Reorder Required",
    inventory_df["Reorder Quantity"],
    0
)

print("REORDER QUANTITIES CALCULATED!")

display(
    inventory_df[
        [
            "Store ID",
            "Product ID",
            "Inventory Level",
            "Average Daily Demand",
            "Safety Stock",
            "Reorder Point",
            "Reorder Quantity",
            "Inventory Status"
        ]
    ].head(15)
)


REORDER QUANTITIES CALCULATED!


,Store ID,Product ID,Inventory Level,Average Daily Demand,Safety Stock,Reorder Point,Reorder Quantity,Inventory Status
73049,S003,P0010,413,100.571429,153.321951,354.464808,0,Healthy
73097,S005,P0018,278,95.428571,302.266481,493.123624,0,Healthy
73025,S002,P0006,160,124.000000,166.090894,414.090894,874,Reorder Required
73026,S002,P0007,53,156.428571,336.602311,649.459454,1379,Reorder Required
73027,S002,P0008,218,148.857143,306.939706,604.653991,1131,Reorder Required
73028,S002,P0009,62,165.428571,328.729411,659.586554,1425,Reorder Required
73029,S002,P0010,419,141.000000,348.325903,630.325903,0,Healthy
73030,S002,P0011,156,80.714286,181.793334,343.221906,591,Reorder Required
73031,S002,P0012,193,126.571429,188.297015,441.439872,881,Reorder Required
73032,S002,P0013,472,163.142857,217.685089,543.970803,0,Healthy


In [9]:
from pathlib import Path

In [10]:

recommendations = inventory_df[
    [
        "Date",
        "Store ID",
        "Product ID",
        "Category",
        "Inventory Level",
        "Average Daily Demand",
        "Stock Coverage",
        "Safety Stock",
        "Reorder Point",
        "Reorder Quantity",
        "Inventory Status"
    ]
].copy()

recommendations["Safety Stock"] = recommendations["Safety Stock"].round(2)
recommendations["Reorder Point"] = recommendations["Reorder Point"].round(2)
recommendations["Average Daily Demand"] = recommendations["Average Daily Demand"].round(2)
recommendations["Stock Coverage"] = recommendations["Stock Coverage"].round(2)

output_path = "../reports/inventory_recommendations.csv"

Path("../reports").mkdir(parents=True, exist_ok=True)

recommendations.to_csv(output_path, index=False)

print("INVENTORY RECOMMENDATIONS EXPORTED!")
print("Total products:", len(recommendations))
print("Products requiring reorder:", (recommendations["Inventory Status"] == "Reorder Required").sum())
print("Healthy products:", (recommendations["Inventory Status"] == "Healthy").sum())
print("Total recommended reorder units:", recommendations["Reorder Quantity"].sum())
print("Saved to:", output_path)

display(recommendations.head(10))


INVENTORY RECOMMENDATIONS EXPORTED!
Total products: 100
Products requiring reorder: 53
Healthy products: 47
Total recommended reorder units: 58097
Saved to: ../reports/inventory_recommendations.csv


,Date,Store ID,Product ID,Category,Inventory Level,Average Daily Demand,Stock Coverage,Safety Stock,Reorder Point,Reorder Quantity,Inventory Status
73049,2024-01-01,S003,P0010,Furniture,413,100.57,4.11,153.32,354.46,0,Healthy
73097,2024-01-01,S005,P0018,Clothing,278,95.43,2.91,302.27,493.12,0,Healthy
73025,2024-01-01,S002,P0006,Toys,160,124.00,1.29,166.09,414.09,874,Reorder Required
73026,2024-01-01,S002,P0007,Toys,53,156.43,0.34,336.60,649.46,1379,Reorder Required
73027,2024-01-01,S002,P0008,Furniture,218,148.86,1.46,306.94,604.65,1131,Reorder Required
73028,2024-01-01,S002,P0009,Electronics,62,165.43,0.37,328.73,659.59,1425,Reorder Required
73029,2024-01-01,S002,P0010,Toys,419,141.00,2.97,348.33,630.33,0,Healthy
73030,2024-01-01,S002,P0011,Groceries,156,80.71,1.93,181.79,343.22,591,Reorder Required
73031,2024-01-01,S002,P0012,Electronics,193,126.57,1.52,188.30,441.44,881,Reorder Required
73032,2024-01-01,S002,P0013,Electronics,472,163.14,2.89,217.69,543.97,0,Healthy
